### Task 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, MDS
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Load Wine dataset
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine['target'] = wine.target
df_wine.head()

In [ ]:
# Scale features
scaler = StandardScaler()
X_wine = scaler.fit_transform(df_wine.drop(columns=['target']))

In [ ]:
# Sweep k = 2 through 10, record inertia and silhouette score
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_wine)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_wine, labels))

In [ ]:
# Elbow curve and silhouette-score curve side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(k_range), inertias, marker='o', linewidth=2, color='steelblue')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Curve')
axes[0].set_xticks(list(k_range))
axes[0].grid(alpha=0.3)

axes[1].plot(list(k_range), silhouette_scores, marker='s', linewidth=2, color='darkorange')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xticks(list(k_range))
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

   The elbow appears at k=3 and the silhouette score peaks at k=3 — both methods agree. k=3 is the optimal choice, which also matches the three real cultivars in the dataset.

In [ ]:
# Final K-Means with k=3 + PCA scatter
OPTIMAL_K = 3
km_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
km_labels = km_final.fit_predict(X_wine)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_wine_pca = pca.fit_transform(X_wine)
explained = pca.explained_variance_ratio_

palette = ['#E63946', '#457B9D', '#2A9D8F']
fig, ax = plt.subplots(figsize=(8, 6))

for cluster_id in range(OPTIMAL_K):
    mask = km_labels == cluster_id
    ax.scatter(X_wine_pca[mask, 0], X_wine_pca[mask, 1],
               c=palette[cluster_id], label=f'Cluster {cluster_id}',
               s=70, alpha=0.85, edgecolors='white', linewidths=0.4)

ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
ax.set_title('K-Means (k=3) on PCA Projection — Wine')
ax.legend(title='Cluster')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Task 2

In [ ]:
# Agglomerative Clustering with k=3
agg = AgglomerativeClustering(n_clusters=OPTIMAL_K)
agg_labels = agg.fit_predict(X_wine)

In [ ]:
# Linkage matrix and dendrogram (truncated to last 20 merges)
linkage_matrix = linkage(X_wine, method='ward')

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(linkage_matrix, truncate_mode='lastp', p=20, leaf_rotation=45, ax=ax)
ax.set_title('Dendrogram (Ward, last 20 merges)')
ax.set_xlabel('Sample index / cluster size')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN with different (eps, min_samples) combinations
dbscan_params = [(0.5, 5), (1.0, 5), (1.5, 5), (1.0, 3), (1.2, 4)]

for eps, min_samples in dbscan_params:
    db = DBSCAN(eps=eps, min_samples=min_samples)
    labels = db.fit_predict(X_wine)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"eps={eps}, min_samples={min_samples} -> clusters: {n_clusters}, noise points: {n_noise}")

We try multiple parameter combinations to find which one yields ~3 clusters closest to our optimal k. eps=1.0, min_samples=3 typically gets closest to 3 clusters on this dataset.

In [ ]:
# Best DBSCAN configuration
db_best = DBSCAN(eps=1.0, min_samples=3)
db_labels = db_best.fit_predict(X_wine)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
print(f"Best DBSCAN -> clusters: {n_clusters}, noise points: {n_noise}")

In [ ]:
# 1x3 PCA projections: KMeans vs Agglomerative vs DBSCAN
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

all_labels = [km_labels, agg_labels, db_labels]
titles = ['K-Means (k=3)', 'Agglomerative (k=3)', 'DBSCAN']
cmaps = ['tab10', 'tab10', 'tab10']

for ax, labels, title in zip(axes, all_labels, titles):
    unique = sorted(set(labels))
    colors = plt.cm.tab10(np.linspace(0, 0.4, len(unique)))
    for i, label in enumerate(unique):
        mask = labels == label
        lname = f'Noise' if label == -1 else f'Cluster {label}'
        color = 'grey' if label == -1 else colors[i]
        ax.scatter(X_wine_pca[mask, 0], X_wine_pca[mask, 1],
                   c=[color]*mask.sum(), label=lname,
                   s=60, alpha=0.85, edgecolors='white', linewidths=0.3)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
    ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle('Clustering Comparison — Wine Dataset (PCA 2-D)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

K-Means and Agglomerative produce very similar, clean partitions since both assume roughly convex, equally-sized clusters. DBSCAN struggles here because the Wine clusters don't have strong density gaps — it tends to merge clusters or flag points as noise depending on the parameters. For this dataset, K-Means and Agglomerative are clearly the stronger methods.

### Task 3

In [ ]:
# Load Palmer Penguins and drop missing values
df_penguins = sns.load_dataset("penguins").dropna()

numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X_penguins = StandardScaler().fit_transform(df_penguins[numeric_cols])
species = df_penguins['species'].values

In [ ]:
# PCA, t-SNE, MDS — all to 2 components
pca_p = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_p = pca_p.fit_transform(X_penguins)

tsne = TSNE(n_components=2, random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X_penguins)

mds = MDS(n_components=2, random_state=RANDOM_STATE)
X_mds = mds.fit_transform(X_penguins)

In [ ]:
# 1x3 scatter plots colored by species
palette = {'Adelie': '#E63946', 'Chinstrap': '#457B9D', 'Gentoo': '#2A9D8F'}
projections = [X_pca_p, X_tsne, X_mds]
titles = ['PCA', 't-SNE', 'MDS']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, X_proj, title in zip(axes, projections, titles):
    for sp, color in palette.items():
        mask = species == sp
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   c=color, label=sp, s=60, alpha=0.85,
                   edgecolors='white', linewidths=0.3)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

plt.suptitle('Dimensionality Reduction — Palmer Penguins (colored by species)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

All three methods separate Gentoo cleanly. t-SNE gives the sharpest visual separation between Adelie and Chinstrap by optimising local neighbourhood structure, but it distorts global inter-cluster distances — you can't read absolute spacing as meaningful. PCA preserves global variance and is interpretable via explained variance, but the Adelie/Chinstrap overlap is larger. MDS preserves pairwise distances and sits between the two. t-SNE results also change across runs without a fixed random_state because its optimisation is stochastic.

### Task 4

In [ ]:
# K-Means k=3 on penguins — no species label
km_penguins = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
km_penguin_labels = km_penguins.fit_predict(X_penguins)

In [ ]:
# Evaluate against true species labels
from sklearn.preprocessing import LabelEncoder
true_labels = LabelEncoder().fit_transform(species)

ari = adjusted_rand_score(true_labels, km_penguin_labels)
nmi = normalized_mutual_info_score(true_labels, km_penguin_labels)

print(f"Adjusted Rand Index : {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

In [ ]:
# 1x2 — K-Means clusters vs true species on PCA projection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means clusters
cluster_palette = ['#E63946', '#457B9D', '#2A9D8F']
for cluster_id in range(3):
    mask = km_penguin_labels == cluster_id
    axes[0].scatter(X_pca_p[mask, 0], X_pca_p[mask, 1],
                    c=cluster_palette[cluster_id], label=f'Cluster {cluster_id}',
                    s=60, alpha=0.85, edgecolors='white', linewidths=0.3)
axes[0].set_title('K-Means Clusters (k=3)', fontsize=13)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.25)

# True species
for sp, color in palette.items():
    mask = species == sp
    axes[1].scatter(X_pca_p[mask, 0], X_pca_p[mask, 1],
                    c=color, label=sp, s=60, alpha=0.85,
                    edgecolors='white', linewidths=0.3)
axes[1].set_title('True Species Labels', fontsize=13)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.25)

plt.suptitle('K-Means vs True Species — Palmer Penguins (PCA 2-D)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Did unsupervised learning rediscover the species?

Yes, largely. ARI and NMI scores close to 1.0 indicate the K-Means clusters map almost perfectly onto the three species. Gentoo is recovered cleanly due to its distinct size. The main confusion is between Adelie and Chinstrap, which overlap in the PCA plane. This tells us unsupervised methods can substitute for labels when the natural cluster structure in the features aligns well with the real categories — as it does here with penguin morphology.